<a href="https://colab.research.google.com/github/UNCL3LO/Netflix-Data-Cleaning/blob/main/Netflix_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from google.colab import files

In [2]:
uploaded= files.upload()

Saving netflix_titles.csv to netflix_titles.csv


In [4]:
filename= list(uploaded.keys())[0]
df= pd.read_csv(filename)
df.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [5]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [7]:
df.describe()

,release_year
count,8807.000000
mean,2014.180198
std,8.819312
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2019.000000
max,2021.000000


In [13]:
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [11]:
df['cast'].isnull().sum()

np.int64(825)

In [17]:
df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [27]:
print(df[df['date_added'].isnull()][['title']])

                                            title
6066  A Young Doctor's Notebook and Other Stories
6174              Anthony Bourdain: Parts Unknown
6795                                      Frasier
6806                                      Friends
6901                              Gunslinger Girl
7196                                     Kikoriki
7254                          La Familia P. Luche
7406                                        Maron
7847                                 Red vs. Blue
8182                 The Adventures of Figaro Pho


In [29]:
df['date_added']= df['date_added'].astype(str).str.strip()

In [30]:
df['date_added']= pd.to_datetime(df['date_added'])
print(df['date_added'])

0      2021-09-25
1      2021-09-24
2      2021-09-24
3      2021-09-24
4      2021-09-24
          ...    
8802   2019-11-20
8803   2019-07-01
8804   2019-11-01
8805   2020-01-11
8806   2019-03-02
Name: date_added, Length: 8807, dtype: datetime64[ns]


In [33]:
df[df['date_added'].isnull()][['title']]

,title
6066,A Young Doctor's Notebook and Other Stories
6174,Anthony Bourdain: Parts Unknown
6795,Frasier
6806,Friends
6901,Gunslinger Girl
7196,Kikoriki
7254,La Familia P. Luche
7406,Maron
7847,Red vs. Blue
8182,The Adventures of Figaro Pho


In [34]:
df= df.dropna(subset=['date_added'])
df = df.reset_index(drop=True)

# Verify null count is down to 0
print("Missing date_added values remaining:", df['date_added'].isnull().sum())
print("Current dataset shape:", df.shape)

Missing date_added values remaining: 0
Current dataset shape: (8797, 12)


In [57]:
df[df['rating'].isnull()][['title']]

,title
5989,13TH: A Conversation with Oprah Winfrey & Ava ...
6823,Gargantia on the Verdurous Planet
7305,Little Lunch
7529,My Honor Was Loyalty


In [52]:
df['rating'].unique()

array(['PG-13', 'TV-MA', 'PG', 'TV-14', 'TV-PG', 'TV-Y', 'TV-Y7', 'R',
       'TV-G', 'G', 'NC-17', '74 min', '84 min', '66 min', 'NR', nan,
       'TV-Y7-FV', 'UR'], dtype=object)

In [71]:
odd_ratings=('74 min', '84 min', '66 min')
df[df['rating'].isin(odd_ratings)][['title','rating','duration']]

,title,rating,duration
5541,Louis C.K. 2017,74 min,NaN
5794,Louis C.K.: Hilarious,84 min,NaN
5813,Louis C.K.: Live at the Comedy Store,66 min,NaN


In [72]:
import numpy as np

In [73]:
mask =df['rating'].str.contains('min', na=False)
df.loc[mask, 'duration']= df.loc[mask,'rating']
df.loc[mask,'rating']= np.nan
df['rating']= df['rating'].fillna(df['rating'].mode()[0])

In [80]:
df['rating'].isin(odd_ratings).sum()

np.int64(0)

In [75]:
df['rating'].str.contains('min').sum()

np.int64(0)

In [78]:
df['duration'].isnull().sum()

np.int64(0)

In [86]:
df['duration'].str.contains('min').sum()

np.int64(6131)

In [88]:
df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2624
cast,825
country,830
date_added,0
release_year,0
rating,0
duration,0


In [89]:
df['director']= df['director'].fillna('unknown')
df['cast']=df['cast'].fillna('unknown')
df['country']= df['country'].fillna('unknown')

print(df.isnull().sum())

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64


In [90]:
df.dtypes

,0
show_id,object
type,object
title,object
director,object
cast,object
country,object
date_added,datetime64[ns]
release_year,int64
rating,object
duration,object


In [91]:
df.duplicated().sum()

np.int64(0)

In [93]:
df['duration_int']= df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_type'] = df['duration'].str.extract(r'([a-zA-Z]+)')

In [95]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_int,duration_type
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,min
1,s2,TV Show,Blood & Water,unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2.0,Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,1.0,Season
3,s4,TV Show,Jailbirds New Orleans,unknown,unknown,unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",1.0,Season
4,s5,TV Show,Kota Factory,unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2.0,Seasons
